# Validation System for CODE_BLOCK Exercises

This notebook demonstrates how to implement validation functions for CODE_BLOCK exercises in the lab.

## 1. Core Validation Framework

In [ ]:
import inspect
import ast
import json
from typing import Any, Tuple, List, Dict, Callable
from datetime import datetime
import hashlib
from IPython.display import display, HTML, Javascript
import ipywidgets as widgets
from IPython.core.magic import register_cell_magic
import sys
from io import StringIO

class ValidationResult:
    """Represents the result of a validation check"""
    def __init__(self, passed: bool, message: str, hints: List[str] = None, details: Dict = None):
        self.passed = passed
        self.message = message
        self.hints = hints or []
        self.details = details or {}
        self.timestamp = datetime.now()
    
    def display(self):
        """Display validation result with formatting"""
        if self.passed:
            status_html = '<div style="padding: 10px; background-color: #d4edda; border: 1px solid #c3e6cb; border-radius: 5px; margin: 10px 0;">'
            status_html += '<h3 style="color: #155724;">✅ Validation Passed!</h3>'
            status_html += f'<p style="color: #155724;">{self.message}</p>'
        else:
            status_html = '<div style="padding: 10px; background-color: #f8d7da; border: 1px solid #f5c6cb; border-radius: 5px; margin: 10px 0;">'
            status_html += '<h3 style="color: #721c24;">❌ Validation Failed</h3>'
            status_html += f'<p style="color: #721c24;">{self.message}</p>'
            
            if self.hints:
                status_html += '<h4 style="color: #856404;">💡 Hints:</h4>'
                status_html += '<ul style="color: #856404;">'
                for hint in self.hints:
                    status_html += f'<li>{hint}</li>'
                status_html += '</ul>'
        
        status_html += '</div>'
        display(HTML(status_html))

class LabProgress:
    """Tracks student progress through the lab"""
    def __init__(self):
        self.completed_blocks = set()
        self.attempts = {}
        self.start_time = datetime.now()
    
    def record_attempt(self, block_num: int, passed: bool):
        if block_num not in self.attempts:
            self.attempts[block_num] = []
        self.attempts[block_num].append({
            'passed': passed,
            'timestamp': datetime.now()
        })
        if passed:
            self.completed_blocks.add(block_num)
    
    def get_progress_percentage(self, total_blocks: int = 19) -> float:
        return (len(self.completed_blocks) / total_blocks) * 100
    
    def display_progress(self, total_blocks: int = 19):
        percentage = self.get_progress_percentage(total_blocks)
        completed = len(self.completed_blocks)
        
        # Create progress bar
        progress_html = f'''
        <div style="margin: 20px 0;">
            <h3>📊 Your Progress: {completed}/{total_blocks} CODE_BLOCKs completed ({percentage:.1f}%)</h3>
            <div style="width: 100%; background-color: #e0e0e0; border-radius: 10px; overflow: hidden;">
                <div style="width: {percentage}%; background-color: #4CAF50; height: 30px; text-align: center; line-height: 30px; color: white;">
                    {percentage:.1f}%
                </div>
            </div>
        </div>
        '''
        display(HTML(progress_html))

# Global progress tracker
lab_progress = LabProgress()

## 2. Validation Functions for Each CODE_BLOCK

In [ ]:
# Validation for CODE_BLOCK_1: PDF Processing
def validate_code_block_1(pdf_document) -> ValidationResult:
    """Validates that student correctly opened a PDF from bytes"""
    try:
        # Check if it's a PyMuPDF document object
        if not hasattr(pdf_document, 'page_count'):
            return ValidationResult(
                False, 
                "The pdf_document doesn't appear to be a valid PyMuPDF Document object.",
                hints=[
                    "Make sure you're using pymupdf.open()",
                    "The first parameter should be 'pdf', which contains the PDF bytes",
                    "Check the import: import pymupdf"
                ]
            )
        
        # Check if document has pages
        if pdf_document.page_count == 0:
            return ValidationResult(
                False,
                "The PDF document has no pages.",
                hints=["Verify that the PDF was downloaded correctly"]
            )
        
        return ValidationResult(
            True,
            f"Perfect! PDF opened successfully with {pdf_document.page_count} pages.",
            details={'page_count': pdf_document.page_count}
        )
    except Exception as e:
        return ValidationResult(
            False,
            f"Error during validation: {str(e)}",
            hints=["Check if pdf_document is defined", "Ensure pymupdf is imported"]
        )

# Validation for CODE_BLOCK_2: Image Extraction
def validate_code_block_2(extract_images_func) -> ValidationResult:
    """Validates the image extraction function"""
    try:
        # Check if it's a callable function
        if not callable(extract_images_func):
            return ValidationResult(
                False,
                "extract_images should be a function.",
                hints=["Make sure you defined the function with 'def extract_images(...):"]
            )
        
        # Check function signature
        sig = inspect.signature(extract_images_func)
        params = list(sig.parameters.keys())
        
        if len(params) < 2:
            return ValidationResult(
                False,
                "Function should accept at least 2 parameters (pdf_document, output_folder).",
                hints=[
                    "Check your function definition",
                    "It should be: def extract_images(pdf_document, output_folder, dpi=150):"
                ]
            )
        
        # Test with mock data
        import tempfile
        import os
        with tempfile.TemporaryDirectory() as tmpdir:
            # We can't fully test without a real PDF, but check if function runs
            try:
                # Check if function accesses expected attributes
                source = inspect.getsource(extract_images_func)
                expected_patterns = ['render', 'get_pixmap', 'save', 'page_number']
                missing_patterns = [p for p in expected_patterns if p not in source]
                
                if missing_patterns:
                    return ValidationResult(
                        False,
                        f"Function seems to be missing key operations: {missing_patterns}",
                        hints=[
                            "Make sure to iterate through pages",
                            "Use page.get_pixmap(dpi=dpi) to render",
                            "Save with pixmap.save()"
                        ]
                    )
                
                return ValidationResult(
                    True,
                    "Great! Your image extraction function looks correct."
                )
                
            except Exception as e:
                return ValidationResult(
                    False,
                    f"Error analyzing function: {str(e)}",
                    hints=["Make sure the function is properly defined"]
                )
                
    except Exception as e:
        return ValidationResult(
            False,
            f"Validation error: {str(e)}",
            hints=["Check if extract_images is defined"]
        )

# Validation for CODE_BLOCK_3: MongoDB Insertion
def validate_code_block_3(insert_result) -> ValidationResult:
    """Validates MongoDB bulk insert operation"""
    try:
        # Check if it's an InsertManyResult
        if not hasattr(insert_result, 'inserted_ids'):
            return ValidationResult(
                False,
                "Result doesn't appear to be from insert_many().",
                hints=[
                    "Use collection.insert_many(documents)",
                    "Make sure to store the result"
                ]
            )
        
        # Check number of inserted documents
        num_inserted = len(insert_result.inserted_ids)
        if num_inserted == 0:
            return ValidationResult(
                False,
                "No documents were inserted.",
                hints=["Check if your documents list is empty"]
            )
        
        return ValidationResult(
            True,
            f"Excellent! Successfully inserted {num_inserted} documents into MongoDB.",
            details={'documents_inserted': num_inserted}
        )
        
    except Exception as e:
        return ValidationResult(
            False,
            f"Validation error: {str(e)}",
            hints=["Make sure insert_result is defined", "Check MongoDB connection"]
        )

## 3. Interactive Validation Widget

In [ ]:
def create_validation_widget(block_num: int, validator_func: Callable, variable_name: str):
    """Creates an interactive validation widget for a CODE_BLOCK"""
    
    button = widgets.Button(
        description=f'Validate CODE_BLOCK_{block_num}',
        button_style='primary',
        icon='check'
    )
    
    output = widgets.Output()
    
    def on_click(b):
        with output:
            output.clear_output()
            
            # Get the variable from the notebook's namespace
            try:
                var = get_ipython().user_ns.get(variable_name)
                if var is None:
                    result = ValidationResult(
                        False,
                        f"Variable '{variable_name}' not found.",
                        hints=[f"Make sure you've run the CODE_BLOCK_{block_num} cell and created '{variable_name}'"]
                    )
                else:
                    result = validator_func(var)
                    lab_progress.record_attempt(block_num, result.passed)
                
                result.display()
                
                # Show progress if passed
                if result.passed:
                    lab_progress.display_progress()
                    
            except Exception as e:
                result = ValidationResult(
                    False,
                    f"Error during validation: {str(e)}",
                    hints=["Check your code for syntax errors"]
                )
                result.display()
    
    button.on_click(on_click)
    
    return widgets.VBox([button, output])

# Example usage in the lab notebook:
# After CODE_BLOCK_1, add:
# display(create_validation_widget(1, validate_code_block_1, 'pdf_document'))

## 4. Cell Magic for Automatic Validation

In [ ]:
@register_cell_magic
def validate_block(line, cell):
    """Cell magic that automatically validates code blocks"""
    # Parse the block number from the line
    parts = line.strip().split()
    if not parts:
        print("Usage: %%validate_block <number>")
        return
    
    block_num = int(parts[0])
    
    # Execute the cell code
    get_ipython().run_cell(cell)
    
    # Map block numbers to validators
    validators = {
        1: (validate_code_block_1, 'pdf_document'),
        2: (validate_code_block_2, 'extract_images'),
        3: (validate_code_block_3, 'insert_result'),
        # Add more validators as needed
    }
    
    if block_num in validators:
        validator_func, var_name = validators[block_num]
        var = get_ipython().user_ns.get(var_name)
        
        if var is not None:
            result = validator_func(var)
            result.display()
            lab_progress.record_attempt(block_num, result.passed)
            
            if result.passed:
                lab_progress.display_progress()
        else:
            print(f"⚠️ Variable '{var_name}' not found after executing the cell.")

# Usage in lab notebook:
# %%validate_block 1
# pdf_document = pymupdf.open("pdf", pdf)

## 5. Advanced Validation Techniques

In [ ]:
class CodeBlockValidator:
    """Advanced validation with output capture and AST analysis"""
    
    @staticmethod
    def validate_with_output(code: str, expected_output_pattern: str = None) -> ValidationResult:
        """Validates code by running it and checking output"""
        old_stdout = sys.stdout
        sys.stdout = captured_output = StringIO()
        
        try:
            exec(code, get_ipython().user_ns)
            output = captured_output.getvalue()
            
            if expected_output_pattern and expected_output_pattern not in output:
                return ValidationResult(
                    False,
                    f"Output doesn't match expected pattern.",
                    hints=[f"Expected to see: {expected_output_pattern}"]
                )
            
            return ValidationResult(True, "Code executed successfully!")
            
        except Exception as e:
            return ValidationResult(
                False,
                f"Code execution failed: {str(e)}",
                hints=["Check for syntax errors", "Ensure all variables are defined"]
            )
        finally:
            sys.stdout = old_stdout
    
    @staticmethod
    def validate_code_structure(code: str, required_elements: List[str]) -> ValidationResult:
        """Validates code structure using AST"""
        try:
            tree = ast.parse(code)
            code_elements = []
            
            for node in ast.walk(tree):
                if isinstance(node, ast.FunctionDef):
                    code_elements.append(f"function:{node.name}")
                elif isinstance(node, ast.ClassDef):
                    code_elements.append(f"class:{node.name}")
                elif isinstance(node, ast.Import):
                    for alias in node.names:
                        code_elements.append(f"import:{alias.name}")
            
            missing = [elem for elem in required_elements if elem not in code_elements]
            
            if missing:
                return ValidationResult(
                    False,
                    f"Missing required elements: {missing}",
                    hints=["Check that you've included all necessary imports and definitions"]
                )
            
            return ValidationResult(True, "Code structure is correct!")
            
        except SyntaxError as e:
            return ValidationResult(
                False,
                f"Syntax error: {str(e)}",
                hints=["Check for missing colons, parentheses, or indentation errors"]
            )

## 6. Progress Dashboard

In [ ]:
def create_progress_dashboard():
    """Creates an interactive progress dashboard"""
    
    # Define all CODE_BLOCKs and their descriptions
    code_blocks = [
        (1, "Open PDF from bytes"),
        (2, "Extract images from PDF"),
        (3, "Insert documents to MongoDB"),
        (4, "Create vector search index"),
        (5, "Build search pipeline"),
        (6, "Execute aggregation"),
        (7, "Define semantic search tool"),
        (8, "Define query tool"),
        (9, "Create function declarations"),
        (10, "Configure LLM with tools"),
        (11, "Implement tool selection"),
        (12, "Execute retrieval tool"),
        (13, "Create session index"),
        (14, "Insert chat messages"),
        (15, "Query chat history"),
        (16, "Add memory to agent (1/4)"),
        (17, "Add memory to agent (2/4)"),
        (18, "Add memory to agent (3/4)"),
        (19, "Add memory to agent (4/4)")
    ]
    
    dashboard_html = """
    <style>
        .dashboard-container {
            font-family: Arial, sans-serif;
            max-width: 800px;
            margin: 20px auto;
        }
        .block-item {
            display: flex;
            align-items: center;
            margin: 10px 0;
            padding: 10px;
            border-radius: 5px;
            background-color: #f8f9fa;
        }
        .block-completed {
            background-color: #d4edda;
            border: 1px solid #c3e6cb;
        }
        .block-number {
            font-weight: bold;
            margin-right: 10px;
            min-width: 40px;
        }
        .block-status {
            margin-left: auto;
            font-size: 20px;
        }
        .stats-container {
            display: flex;
            justify-content: space-around;
            margin: 20px 0;
            padding: 20px;
            background-color: #e9ecef;
            border-radius: 10px;
        }
        .stat-item {
            text-align: center;
        }
        .stat-number {
            font-size: 36px;
            font-weight: bold;
            color: #007bff;
        }
        .stat-label {
            color: #6c757d;
        }
    </style>
    
    <div class="dashboard-container">
        <h2>📊 Lab Progress Dashboard</h2>
        
        <div class="stats-container">
            <div class="stat-item">
                <div class="stat-number">{completed}</div>
                <div class="stat-label">Completed</div>
            </div>
            <div class="stat-item">
                <div class="stat-number">{percentage:.0f}%</div>
                <div class="stat-label">Progress</div>
            </div>
            <div class="stat-item">
                <div class="stat-number">{remaining}</div>
                <div class="stat-label">Remaining</div>
            </div>
        </div>
        
        <h3>CODE_BLOCK Status</h3>
        {blocks_html}
    </div>
    """
    
    blocks_html = ""
    for num, desc in code_blocks:
        is_completed = num in lab_progress.completed_blocks
        status_icon = "✅" if is_completed else "⭕"
        css_class = "block-item block-completed" if is_completed else "block-item"
        
        blocks_html += f"""
        <div class="{css_class}">
            <span class="block-number">#{num}</span>
            <span class="block-description">{desc}</span>
            <span class="block-status">{status_icon}</span>
        </div>
        """
    
    completed = len(lab_progress.completed_blocks)
    total = len(code_blocks)
    percentage = (completed / total) * 100 if total > 0 else 0
    remaining = total - completed
    
    final_html = dashboard_html.format(
        completed=completed,
        percentage=percentage,
        remaining=remaining,
        blocks_html=blocks_html
    )
    
    display(HTML(final_html))

# Add this to the lab notebook:
# create_progress_dashboard()

## 7. Example Integration in Lab Notebook

In [ ]:
# This shows how to integrate validation in the actual lab notebook

# First, load the validation system at the beginning of the lab
display(HTML("""
<div style="background-color: #e7f3ff; border-left: 6px solid #2196F3; padding: 10px; margin: 10px 0;">
    <h3>🎯 Interactive Validation System Loaded!</h3>
    <p>After completing each CODE_BLOCK, click the validation button to check your work.</p>
    <p>You'll receive immediate feedback and hints if needed.</p>
</div>
"""))

# Then after each CODE_BLOCK in the lab, add a validation cell like this:
print("Example: After CODE_BLOCK_1 in the lab notebook, add:")
print("""```python
# 🧪 Validate CODE_BLOCK_1
display(create_validation_widget(1, validate_code_block_1, 'pdf_document'))
```""")

print("\nOr use the cell magic approach:")
print("""```python
%%validate_block 1
pdf_document = pymupdf.open("pdf", pdf)
```""")

## 8. Achievement System

In [ ]:
class AchievementSystem:
    """Gamification through achievements"""
    
    def __init__(self):
        self.achievements = {
            'first_step': {'name': '🎉 First Steps', 'desc': 'Complete your first CODE_BLOCK', 'blocks_needed': 1},
            'pdf_master': {'name': '📄 PDF Master', 'desc': 'Complete all PDF processing blocks', 'blocks_needed': [1, 2]},
            'data_engineer': {'name': '💾 Data Engineer', 'desc': 'Successfully store data in MongoDB', 'blocks_needed': [3]},
            'search_wizard': {'name': '🔍 Search Wizard', 'desc': 'Implement vector search', 'blocks_needed': [4, 5, 6]},
            'tool_builder': {'name': '🛠️ Tool Builder', 'desc': 'Create all agent tools', 'blocks_needed': [7, 8, 9]},
            'ai_engineer': {'name': '🤖 AI Engineer', 'desc': 'Connect LLM with tools', 'blocks_needed': [10, 11, 12]},
            'memory_keeper': {'name': '🧠 Memory Keeper', 'desc': 'Implement conversation memory', 'blocks_needed': [13, 14, 15, 16, 17, 18, 19]},
            'lab_champion': {'name': '🏆 Lab Champion', 'desc': 'Complete all CODE_BLOCKs!', 'blocks_needed': list(range(1, 20))}
        }
        self.earned = set()
    
    def check_achievements(self, completed_blocks: set):
        """Check for newly earned achievements"""
        newly_earned = []
        
        for ach_id, ach_data in self.achievements.items():
            if ach_id not in self.earned:
                blocks_needed = ach_data['blocks_needed']
                
                if isinstance(blocks_needed, int):
                    # Achievement based on number of blocks
                    if len(completed_blocks) >= blocks_needed:
                        self.earned.add(ach_id)
                        newly_earned.append(ach_data)
                else:
                    # Achievement based on specific blocks
                    if all(b in completed_blocks for b in blocks_needed):
                        self.earned.add(ach_id)
                        newly_earned.append(ach_data)
        
        # Display newly earned achievements
        for achievement in newly_earned:
            self._display_achievement(achievement)
    
    def _display_achievement(self, achievement: dict):
        """Display achievement notification"""
        html = f"""
        <div style="
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 20px;
            border-radius: 10px;
            margin: 20px 0;
            text-align: center;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
        ">
            <h2 style="margin: 0; font-size: 48px;">🎊</h2>
            <h3 style="margin: 10px 0;">Achievement Unlocked!</h3>
            <h2 style="margin: 10px 0;">{achievement['name']}</h2>
            <p style="margin: 5px 0; opacity: 0.9;">{achievement['desc']}</p>
        </div>
        """
        display(HTML(html))

# Global achievement system
achievements = AchievementSystem()

## Summary

This validation system provides:

1. **Immediate Feedback**: Students know instantly if their solution is correct
2. **Helpful Hints**: Targeted hints guide students toward the solution
3. **Progress Tracking**: Visual progress bars and dashboards
4. **Gamification**: Achievements make the lab more engaging
5. **Multiple Integration Options**:
   - Interactive widgets
   - Cell magic
   - Manual validation functions

To integrate into your lab:
1. Load this validation notebook at the start
2. Add validation widgets after each CODE_BLOCK
3. Use the progress dashboard to motivate students
4. Celebrate completions with achievements